In [ ]:
import os
import re
import sys
import logging
import json
import numpy as np
import pandas as pd
import pickle
import xgboost as xgb
import plotly.graph_objects as go
from collections import defaultdict
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
#from utils.db_interface import get_engine
from typing import Optional
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import sys
import joblib
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine



sys.path.append("..")
DATABASE_URI = "postgresql+psycopg2://USER@localhost:5432/mimic"

# ----------------------
# Configuration
# ----------------------
MODELS_DIR = "../../../models/calibrated"
table_name = "merged_mix_features"
PRED_DIR ="../../../models/calibrated/preds"
SUBANALYSIS_DIR = "../results"
LOG_LEVEL = logging.INFO
# ----------------------
# Set up logging
# ----------------------
logging.basicConfig(
    level=LOG_LEVEL,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
# -----
# DB connection
)
DATABASE_URI = "postgresql+psycopg2://USER@localhost:5432/mimic"

ValueError: Unrecognised argument(s): DATABASE_URI

## Helper functions

In [ ]:
def load_and_prepare_data_xgb(table_name):
    
    cache = {} 
    if table_name in cache:
        logging.info(f"🔁 Using cached data for {table_name} ")
        return cache[table_name]

    logging.info(f" Loading val/test data from ce_approach.{table_name}")
    engine = get_engine()

    df = pd.read_sql(f"""
        SELECT * FROM ce_approach.{table_name}
        WHERE split IN ('val', 'test')
    """, engine)

    # Vorverarbeitung
    # IDs und Kontext separat sichern
    id_cols = ["subject_id", "icustay_id", "context_start", "context_end"]
    id_df = df[id_cols + ["split"]].copy()

    # Label engineering
    df["label"] = df["positive_event"].astype(int)

    excluded = {"positive_event", "positive_sample", "split", "label", "treatment_given", "only_2_values"} | set(id_cols)
    
    logging.info(f"Features excluded: {excluded}")

    feature_cols = [c for c in df.columns if c not in excluded]

    val_df = df[df["split"] == "val"]
    test_df = df[df["split"] == "test"]

    x_val = val_df[feature_cols]
    y_val = val_df["label"]
    x_test = test_df[feature_cols]
    y_test = test_df["label"]

    # Reihenfolge:
    # Die IDs passend zum Split filtern
    id_val = id_df[id_df["split"] == "val"].reset_index(drop=True)
    id_test = id_df[id_df["split"] == "test"].reset_index(drop=True)

    logging.info(f"✅ Data loaded and cached for {table_name}")

    cache[table_name] = (x_val, y_val, x_test, y_test, feature_cols, id_val, id_test)
    return cache[table_name]

In [ ]:
def evaluate_xgboost_model(model_name: str, x_val, x_test):
    model_path = os.path.join(MODELS_DIR, model_name)
    logging.info(f"📦 Loading XGBoost model from: {model_path}")
    bst = xgb.Booster()
    bst.load_model(model_path)

    dval = xgb.DMatrix(x_val, missing=np.nan)
    dtest = xgb.DMatrix(x_test, missing=np.nan)

    y_val_pred = bst.predict(dval)
    y_test_pred = bst.predict(dtest)

    return y_val_pred, y_test_pred, bst



In [ ]:
def evaluate_calibrated_model(model_name: str, x_val: pd.DataFrame, x_test: pd.DataFrame, threshold: float = 0.5):
    """
    Lädt ein kalibriertes XGBoost-Modell im .pkl-Format und gibt Vorhersagen für Validierung und Testdaten zurück.

    Args:
        model_name (str): Dateiname des Modells (inkl. .pkl).
        x_val (pd.DataFrame): Validierungsdaten.
        x_test (pd.DataFrame): Testdaten.
        threshold (float): Schwelle für binäre Vorhersagen (default: 0.5).

    Returns:
        Tuple[np.ndarray, np.ndarray, CalibratedXGBModel]:
            - y_val_pred: binäre Vorhersagen für Validierungsdaten
            - y_test_pred: binäre Vorhersagen für Testdaten
            - model: das geladene CalibratedXGBModel
    """
    model_path = os.path.join(MODELS_DIR, model_name)
    logging.info(f"📦 Lade kalibriertes Modell von: {model_path}")
 
    calibrated_model = joblib.load(os.path.join(MODELS_DIR, f"{model_name}.pkl"))
    #y_val_pred = model.predict(x_val, threshold=threshold)
    y_val_pred = calibrated_model.predict_proba(x_val)
    y_test_pred = calibrated_model.predict_proba(x_test)

    return y_val_pred, y_test_pred, calibrated_model

In [ ]:
def classify_last_map(map_json_list):
    #print("I got:", map_json_list)
    #print("Type:", type(map_json_list))
    try:
        last_value = float(map_json_list[-1]['value'])
        #print(last_value)
        if last_value <= 65:
            return 1
        elif 65 < last_value <= 70:
            return 2
        elif 70 < last_value <= 100:
            return 3
        elif last_value > 100:
            return 4
        else:
            return 0  
    except:
        print("problem!!")
        return 0  # Parsing error oder ungültig

In [ ]:
# Test classify_last_map
# engine = get_engine()
# map_value_df = pd.read_sql(f"""
#     SELECT m.icustay_id, m.context_start, m.map_values
#     FROM ce_approach.mix_windows m
#     JOIN ce_approach.split_all_subjects s ON m.subject_id = s.subject_id
#     WHERE s.split = 'test'
# """, engine)


# map_value_df.head()

# #map_value_df["map_values"].apply(classify_last_map)
# test_input = map_value_df["map_values"].iloc[544]
# print("Test input:", test_input)
# print("Test output:", classify_last_map(test_input))


In [ ]:
# Neue Spalte erstellen: 0 = keine Infos oder alles 0, 1 = mindestens ein Risiko, 2 = hohes Risiko (>=2)
def classify_hypotension_risk(count):
    if pd.isna(count):  # komplett fehlende Daten
        return 0
    elif count >= 2:
        return 2
    elif count == 1:
        return 1
    else:
        return 0

In [ ]:
# TEST classify_hypotension_risk

# >>>>> Risikofaktor-Klassifikation <<<<<
# risk_factor_cols = [
#     "obesity",
#     "hypertension",
#     "diabetes",
#     "kidney_disease",
#     "lung_disease",
#     "heart_disease",
#     "drug_abuse"
# ]

    

#     # Test classify_hypotension_risk
# engine = get_engine()
# df = pd.read_sql(f"""
#     SELECT *
#     FROM ce_approach.merged_mix_features 
#     WHERE split = 'test'
# """, engine)


# #print(df.head())
# # Anzahl der vorhandenen Risikoerkrankungen pro Zeile zählen
# df["risk_factor_count"] = df[risk_factor_cols].sum(axis=1)
# print(df["risk_factor_count"].unique())
# test_input = df["risk_factor_count"].iloc[1]
# print("Test input:", test_input)
# print("Test output:", classify_hypotension_risk(test_input))

In [ ]:
def compute_treatment_distances(df):
    #print("before", len(df))
    df = df.loc[df["positive_sample"] == True]
    #print("after", len(df))

    results = []
    for icu_id, group in df.groupby("icustay_id"):
        group = group.sort_values("context_start").copy()

        # treatment times for this ICU stay where positive_event = True
        treatment_times = group.loc[group["positive_event"] == 1, "treatment_starttime"].dropna().values
        #print("treatment times:", treatment_times)
        if len(treatment_times) == 0:
            # kein Treatment -> Distanzfelder NaN
            group["treatment_distance"] = np.nan
            group["treatment_distance_future"] = np.nan
            group["treatment_distance_past"] = np.nan
        else:
            distances = []      # absolute distance to nearest treatment
            dist_future = []    # distance to next treatment in future
            dist_past = []      # distance to last treatment in past

            for context_start in group["context_start"].values:
                # differences in minutes
                diffs = (treatment_times - context_start) / np.timedelta64(1, "m")

                # future: smallest positive difference
                fut = diffs[diffs > 0].min() if np.any(diffs > 0) else np.nan
                # past: largest negative difference
                pas = diffs[diffs < 0].max() if np.any(diffs < 0) else np.nan
                # absolute minimum
                abs_min = np.nanmin([abs(fut) if not np.isnan(fut) else np.inf,
                                     abs(pas) if not np.isnan(pas) else np.inf])
                if abs_min == np.inf:
                    abs_min = np.nan

                distances.append(abs_min)
                dist_future.append(fut)
                dist_past.append(pas)

            group["treatment_distance"] = distances
            group["treatment_distance_future"] = dist_future
            group["treatment_distance_past"] = dist_past

           # binning
            for col in ["treatment_distance_future", "treatment_distance_past", "treatment_distance"]:
                values = group[col]
                if col == "treatment_distance_past":
                    values = values.abs()  # minutes ago as positive

                # initial cut (without NaNs)
                binned = pd.cut(
                    values,
                    bins=[0, 60, np.inf],    # strictly ≥0
                    labels=[1, 2],           # 1 = ≤60, 2 = >60
                    include_lowest=True
                )

                # add category 0 for NaNs
                binned = binned.cat.add_categories([0])
                binned = binned.fillna(0)

                group[f"{col}_bin"] = binned.astype(int)

        results.append(group)

    return pd.concat(results, axis=0)


In [ ]:
def create_test_full_for_table(table_name: str):
    """
    Erstellt ein DataFrame `test_full` mit Features, Label und Modellvorhersage.
    Zusätzliche features für Subgruppenanalyse: icustay_id, positive sample, positive event, context_start,
    {type}_windows.map_values
    """

    # Modellname und Tabellennamen anpassen
    model_name = f"xgb_{table_name}"
    table_name = f"merged_{table_name}_features"

    # Lade Daten einmal – inkl. y-Test und den indices für val und test
    x_val, y_val, x_test, y_test, feature_cols, id_val, id_test = load_and_prepare_data_xgb(table_name)

    # Lade Modell und Vorhersagen machen

    y_val_pred, y_test_pred, model = evaluate_calibrated_model(model_name, x_val, x_test)

    # Erstelle Test-Full-DataFrame
    test_full = x_test.copy()
    test_full["label"] = y_test.values
    test_full["preds"] = y_test_pred

    # Jetzt: concat id_test (Identifikatoren und context_start) mit test_full (Features, Label, Preds)
    test_full = pd.concat([id_test.reset_index(drop=True), test_full.reset_index(drop=True)], axis=1)


    #### Mehrere Zusatzinformationen für die Subgruppenanalyse aus der originalen tabelle holen + neue erstellen ####
    engine = get_engine()

   # >>>>> Positive_sample, Positive_event, icustayid und contextstart for exact merging of rows<<<<<

    df_additional = pd.read_sql(f"""
        SELECT icustay_id, context_start, positive_sample, positive_event
        FROM ce_approach.{table_name}
        WHERE split = 'test'
    """, engine)

    map_value_tables = {
    "mix": "ce_approach.mix_windows",
    "inv": "ce_approach.invasive_windows",
    "noninv": "ce_approach.noninvasive_windows"
}
    if "mix" in table_name:
        group = "mix"
    elif "inv" in table_name:
        group = "inv"
    elif "noninv" in table_name:
        group = "noninv"
    else:
        raise ValueError(f"Unbekannter Tabellenname: {table_name}")
    
    # >>>>> MAP Range <<<<<
    
    map_value_df = pd.read_sql(f"""
    SELECT m.icustay_id, m.context_start, m.map_values
    FROM {map_value_tables[group]} m
    JOIN ce_approach.split_all_subjects s ON m.subject_id = s.subject_id
    WHERE s.split = 'test'
""", engine)
    
    map_value_df["map_elevation_range"] = map_value_df["map_values"].apply(classify_last_map)
    map_value_df = map_value_df.drop(columns="map_values")  
    
    # Merge der Zusatzinfos (Positive Event, Positive sample, (..), MAP map_elevation_range)
    test_full = test_full.merge(df_additional, on=["icustay_id", "context_start"], how="left")
    test_full = test_full.merge(map_value_df, on=["icustay_id", "context_start"], how="left")

     # >>>>> Risikofaktor-Klassifikation <<<<<
    risk_factor_cols = [
        "obesity",
        "hypertension",
        "diabetes",
        "kidney_disease",
        "lung_disease",
        "heart_disease",
        "drug_abuse"
    ]

    # Anzahl der vorhandenen Risikoerkrankungen pro Zeile zählen
    test_full["risk_factor_count"] = test_full[risk_factor_cols].sum(axis=1)
    # neue Spalte hinzufügen
    test_full["hypotension_risk_factor"] = test_full["risk_factor_count"].apply(classify_hypotension_risk)
    # Count spalte löschen
    test_full.drop(columns="risk_factor_count", inplace=True)

    # >>>>> treatment_distance <<<<<
    # get context window ids für jede Zeile pro icu stay id (this approach doesnt work since we had to skip many context windows due to not enough map values, so
    # we instead take absolute time difference in minutes between context_start and treatment time)
    # get distance of each context window to the closest context windows that was positive event=true 
    # seperate: distance_future and distance_past (+3 = three windows before treatment, -2 = two windows after treatment)
    # run classify_treatment_distance() function to get binned ranges total: (before and after counts the same 1) 5 2) 10) 

    # Create context_window_id: sequential number per icustay_id
    #test_full = test_full.sort_values(["icustay_id", "context_start"])
    #test_full["context_window_id"] = test_full.groupby("icustay_id").cumcount() + 1

    #df_treatment = pd.read_sql(f"""
    #SELECT icustay_id, context_start, context_end, treatment_starttime
    #FROM ce_approach.all_mv_target_window_map_values
    #WHERE icustay_id IN (SELECT icustay_id FROM ce_approach.{table_name} WHERE split='test')
#""", engine)
    
    # Merge into test_full
 #   test_full = test_full.merge(
 #       df_treatment,
 #       on=["icustay_id", "context_start", "context_end"],
 #       how="left"
 #   )

    # anwenden
  #  test_full = compute_treatment_distances(test_full)

    return test_full

In [ ]:
# DataFrames mit Namen als Schlüssel (für späteren Zugriff)
test_full_dfs = {
    "mix_ntg": create_test_full_for_table("mix"),
    "inv_ntg": create_test_full_for_table("inv"),
    "noninv_ntg": create_test_full_for_table("noninv"),
}

In [ ]:
with open("test_full_dfs.pkl", "wb") as f:
    pickle.dump(test_full_dfs, f)